# Persiapan

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
koneksi = sqlite3.connect('praktik3.db')

In [ ]:
kursor = koneksi.cursor()

# NO.1


In [ ]:
#Mengimport data dari excel
ImportDataTabelCustomer = pd.read_excel('Online Retail Store - customer.xlsx')
ImportDataTabelProduct = pd.read_excel('Online Retail Store - product.xlsx')
ImportDataTabelSales = pd.read_excel('Online Retail Store - sales.xlsx')

#Memindahkan data excel ke sql
ImportDataTabelCustomer.to_sql('Customer', koneksi, if_exists='replace')
ImportDataTabelProduct.to_sql('Product', koneksi, if_exists='replace')
ImportDataTabelSales.to_sql('Sales', koneksi, if_exists='replace')

2901

In [ ]:
pd.read_sql_query("""select * from customer;""",koneksi)

,index,CustomerID,Name,Surname,Loyalty_Discount
0,0,100000001,Simon,Walsh,0.10
1,1,100000003,Liam,Brown,0.09
2,2,100000005,Deirdre,Pullman,0.10
3,3,100000007,Dorothy,Thomson,0.07
4,4,100000010,Dominic,Parr,0.00
...,...,...,...,...,...
937,937,400000830,Trevor,Mitchell,0.04
938,938,400000837,Sean,Rampling,0.05
939,939,400000914,Anna,Randall,0.05
940,940,400000915,Melanie,Hemmings,0.01


In [ ]:
pd.read_sql_query("""select * from Product;""",koneksi)

,index,Item_id,Description,Retail_Price
0,0,7645689457,Apron,9.48
1,1,7690258628,Baby Blanket,21.93
2,2,6388591017,Baby Quilt A,100.68
3,3,7502168586,Baby Quilt C,38.06
4,4,8675740287,Baby Quilt B,47.76
...,...,...,...,...
120,120,8864799421,Valance B,31.80
121,121,8241463559,Wallet,49.75
122,122,2926937052,Washcloth B,10.78
123,123,6660530324,Washcloth A,35.14


In [ ]:
pd.read_sql_query("""select * from Sales;""",koneksi)

,index,TransactionID,Order_Date,Arrival_Date,CustomerID,Shipping_State,Item_id,Quantity,Rating
0,0,113252,2003-07-03 00:00:00,2003-07-11 00:00:00,100000007,Alabama,6697166886,2,4
1,1,113248,2003-11-15 00:00:00,2003-11-21 00:00:00,400000706,Alabama,1893735141,5,3
2,2,113208,2004-12-03 00:00:00,2004-12-09 00:00:00,100000121,Alabama,2363094138,1,5
3,3,113138,2004-03-03 00:00:00,2004-03-09 00:00:00,100000091,Alabama,8948840620,9,5
4,4,113047,2004-12-21 00:00:00,2004-12-22 00:00:00,100000332,Alabama,1284609276,9,3
...,...,...,...,...,...,...,...,...,...
2896,2896,110478,2005-05-31 00:00:00,2005-06-01 00:00:00,100000229,Wyomin,2178891798,7,3
2897,2897,110449,2003-05-08 00:00:00,2003-05-14 00:00:00,100000446,Wyomin,9870137717,6,3
2898,2898,110409,2004-11-25 00:00:00,2004-12-03 00:00:00,400000554,Wyomin,9223468295,2,4
2899,2899,110395,2004-02-26 00:00:00,2004-03-01 00:00:00,200000264,Wyomin,9870137717,3,5


In [ ]:
pd.read_sql_query("""
SELECT
Product.retail_price*Sales.quantity
FROM Product
INNER JOIN Sales
ON Product.Item_id = Sales.Item_id;""",koneksi)

,Product.retail_price*Sales.quantity
0,18.96
1,18.96
2,28.44
3,28.44
4,28.44
...,...
2875,798.00
2876,798.00
2877,798.00
2878,798.00


In [ ]:
pd.read_sql_query("""
WITH penjualan_bulanan AS (
  SELECT
    STRFTIME('%Y-%m', Order_Date) AS periode,
    SUM(Quantity * Retail_Price) AS total_sales
  FROM Sales
  JOIN Product ON Sales.Item_id = Product.Item_id
  GROUP BY STRFTIME('%Y-%m', Order_Date)
  ORDER BY STRFTIME('%Y-%m', Order_Date)
)

SELECT
  periode,
  total_sales,
  LAG(total_sales) OVER (ORDER BY periode) AS prev_month_sales,
  IFNULL(ROUND((total_sales - LAG(total_sales) OVER (ORDER BY periode)) / LAG(total_sales) OVER (ORDER BY periode) * 100, 2), 0) AS sales_growth
FROM penjualan_bulanan
ORDER BY periode;
""",koneksi)




,periode,total_sales,prev_month_sales,sales_growth
0,2003-01,9307.88,NaN,0.00
1,2003-02,13228.38,9307.88,42.12
2,2003-03,21297.68,13228.38,61.00
3,2003-04,9657.69,21297.68,-54.65
4,2003-05,23963.28,9657.69,148.13
5,2003-06,13674.98,23963.28,-42.93
6,2003-07,24012.46,13674.98,75.59
7,2003-08,12909.50,24012.46,-46.24
8,2003-09,25439.48,12909.50,97.06
9,2003-10,45836.16,25439.48,80.18


# NO.2

In [ ]:
pd.read_sql_query("""
SELECT
  cust_cat,
  COUNT(*) AS total_user,
  avg_quantity,
  avg(pengeluaran_customer) AS avg_spending

FROM (
    SELECT
      SUM(p.retail_price * s.quantity) OVER (PARTITION BY c.CustomerID) AS pengeluaran_customer,
      avg(s.quantity) OVER (PARTITION BY c.CustomerID) AS avg_quantity,

      CASE
        WHEN SUM(p.retail_price * s.quantity) OVER (PARTITION BY c.CustomerID) > 500 THEN 'High_spender'
        WHEN SUM(p.retail_price * s.quantity) OVER (PARTITION BY c.CustomerID) BETWEEN 300 AND 500 THEN 'Mid_spender'
        ELSE 'Low_spender'
      END AS cust_cat
    FROM Product p
    INNER JOIN Sales s ON p.Item_id = s.Item_id
    INNER JOIN Customer c ON s.CustomerID = c.CustomerID
    WHERE s.Shipping_state NOT IN ('Wyomin','Idaho','Alaska')
  ) AS tabelBoongan
GROUP BY cust_cat

;""", koneksi)

,cust_cat,total_user,avg_quantity,avg_spending
0,High_spender,2269,6.333333,1405.410357
1,Low_spender,218,5.000000,174.855459
2,Mid_spender,221,3.333333,408.755837


# NO. 3


In [ ]:
pd.read_sql_query("""
WITH avg_ratings AS (
    SELECT
        Item_id AS Product,
        AVG(Rating) AS avg_rating
    FROM
        sales
    WHERE
        strftime('%Y', Order_Date) = '2004'
    GROUP BY
        Product
),

rank_produk AS (
    SELECT
        Product,
        avg_rating,
        ROW_NUMBER() OVER(ORDER BY avg_rating DESC) as Ranking
    FROM
        avg_ratings
),

rank5_teratas AS (
    SELECT
        '2004' AS Year,
        Product,
        avg_rating,
        Ranking
    FROM
        rank_produk
    ORDER BY
        Ranking
    LIMIT 5
),

rank5_terbawah AS (
    SELECT
        '2004' AS Year,
        Product,
        avg_rating,
        ROW_NUMBER() OVER(ORDER BY avg_rating) + (SELECT COUNT(*) FROM rank5_teratas) as Ranking
    FROM
        avg_ratings
    ORDER BY
        avg_rating
    LIMIT 5
)

SELECT * FROM rank5_teratas
UNION ALL
SELECT * FROM rank5_terbawah
ORDER BY Ranking;

""", koneksi)

,Year,Product,avg_rating,Ranking
0,2004,9870137717,4.600000,1
1,2004,4681342313,4.500000,2
2,2004,7645888340,4.500000,3
3,2004,6660530324,4.444444,4
4,2004,7270957383,4.428571,5
5,2004,2355020550,3.444444,6
6,2004,5480721202,3.444444,7
7,2004,3495221948,3.500000,8
8,2004,8864799421,3.500000,9
9,2004,9538578209,3.500000,10
